# ==========================================
# Linear Quantile：线性分位数回归
# ==========================================

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

env = 'local'
# env = 'colab'
# env = 'kaggle'

if env == 'kaggle':
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

    micro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/micro_data.parquet')
    macro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/MacroFactor_McCracken.parquet')
    path = '/kaggle/working'

elif env == 'colab':
    import google.colab 
    # 1. 设置环境变量
    google.colab.drive.mount('/content/drive')
    os.environ['KAGGLE_USERNAME'] = 'jianbinchenuc'
    os.environ['KAGGLE_KEY'] = 'fa590c922d3624b8e0734dc7a90dd95a'

    # 2. 安装 kaggle 库 (如果尚未安装)
    !pip install -q kaggle

    # 3. 创建目录并下载数据集
    # -p 参数指定下载目录，--unzip 参数会自动解压
    !mkdir -p /content/mdndata
    !kaggle datasets download -d jianbinchenuc/mdndata-1990 -p /content/mdndata --unzip

    # 4. 验证文件是否已存在
    print("已下载的文件：", os.listdir('/content/mdndata'))
    micro_df = pd.read_parquet('/content/mdndata/micro_data.parquet') # 可以根据需要切换到 Colab 的路径
    macro_df = pd.read_parquet('/content/mdndata/MacroFactor_McCracken.parquet') # 可以根据需要切换到 Colab 的路径
    path = '/content/drive/MyDrive/Colab Notebooks/'

elif env == 'local':
    micro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/micro_data.parquet') # local 路径
    macro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/MacroFactor_McCracken.parquet') # local 路径
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'

else:
    raise ValueError("Invalid environment specified. Please choose 'kaggle', 'colab', or 'local'.")

In [2]:
from pathlib import Path
import importlib.util
import copy
import json
import math
import os
import random
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm, t as student_t

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from datetime import datetime
import time

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

# ==========================================
# 全局随机种子锁死函数
# ==========================================
def seed_everything(seed=42):
    """
    全局锁死随机种子，确保深度学习模型的绝对可复现性
    """
    # 1. 锁死 Python 内置随机库
    random.seed(seed)
    
    # 2. 锁死 Python 哈希表的随机化 (保证字典遍历顺序一致)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. 锁死 Numpy 随机种子
    np.random.seed(seed)
    
    # 4. 锁死 PyTorch CPU 随机种子
    torch.manual_seed(seed)
    
    # 5. 锁死 PyTorch GPU 随机种子
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # 如果你有双卡/多卡
        
        # 【极其关键】锁死 CuDNN 底层算法的随机性
        # 注意：设置为 True 可能会略微牺牲一点点 GPU 训练速度，但为了发论文复现，这是必须的！
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed} to ensure reproducibility.")

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")


Using device: mps


In [3]:
# ==========================================
# 1. 全局参数设置
# ==========================================
run_test = 'quick'  # 'quick' 或 'full'
random_seed = 42

initial_train_months = 180
val_months = 12
test_months = 12

# 初始窗口完整训练；之后每个窗口在原模型参数上继续微调。
early_stopping_max_epochs = 300
first_window_epochs = 100
later_window_epochs = 50
batch_size = 1024
lr = 1e-3
patience = 20
dropout_rate = 0.05
sigma_floor = 0.005  # 0.5% monthly component-scale floor
# quick_max_stocks = 500

normalization_eps = 1e-8

epochs_type = 'early_stopping'        # 'fixed' 或 'early_stopping'
windows_type = 'rolling'     # 'expanding' 或 'rolling'
weights_type = 'month_equal'  # 'uniform', 'month_equal', or 'time_decay'
# FiLM = 'NoFiLM' # 'FiLM' 或 'NoFiLM'
MODEL_TYPE = "fused"  # "fused" 或 "structured"
best_k = 1            # 1 或 3
GLOBAL_DIST_TYPE = "normal"   # 't' 或 'normal'

ALPHAS = (0.01, 0.05, 0.10)
ALPHA_SUFFIX = {0.01: "1", 0.05: "5", 0.10: "10"}

QUANTILE_LEVELS = np.array([
    0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.10, 0.25, 0.50,
    0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.9975, 0.999
], dtype=float)

In [4]:
print('Micro DataFrame shape:', micro_df.shape)
print('Micro months:', micro_df['mthcaldt'].nunique())
display(pd.concat([micro_df.head(5), micro_df.tail(5)]))
print("Macro DataFrame shape:", macro_df.shape)
display(pd.concat([macro_df.head(5), macro_df.tail(5)]))


Micro DataFrame shape: (1560360, 41)
Micro months: 408


,permno,mthcaldt,mthret,mthprc,mthcap,siccd,gvkey,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,ffi49,target_ret_final,MOM12m,mu_bench,sigma_bench,mthcap_log
0,10001,1991-01-31,0.013158,9.625,1.014475e+04,4920.0,012994,0.896385,5.690808,3.307560,0.445982,4.872598,14.070388,0.044401,0.099749,0.138304,0.082956,0.112235,0.112708,0.161306,0.392076,2.430626,0.061202,0.672101,1.796365,0.316439,1.340247,1.166318,8.789413,10.931698,3.261452,0.000000,0.0,0.0,-0.055775,31,0.012987,0.009425,0.002110,0.022682,9.224712
1,10001,1991-02-28,0.012987,9.750,1.027650e+04,4920.0,012994,0.975332,5.299653,3.037383,0.434947,5.205927,14.253121,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,-0.012244,0.029179,0.003717,0.022716,9.237615
2,10001,1991-03-31,-0.012244,9.500,1.001300e+04,4920.0,012994,0.975332,5.299653,2.959502,0.423795,5.072442,13.887656,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,0.039474,0.029348,0.001628,0.022954,9.211640
3,10001,1991-04-30,0.039474,9.875,1.040825e+04,4920.0,012994,0.975332,5.299653,3.115265,0.446100,5.339412,14.618585,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,0.000000,0.016744,0.004917,0.025398,9.250354
4,10001,1991-05-31,0.000000,9.875,1.040825e+04,4920.0,012994,1.040348,5.316239,3.086420,0.444182,6.893394,14.618585,0.047832,0.103923,0.139492,0.083400,0.119963,0.115336,0.165372,0.375721,2.271979,0.059539,0.472059,1.708091,0.318617,1.389858,1.185531,8.331080,10.836074,3.210526,0.000000,0.0,0.0,-0.019640,31,0.079241,0.070429,0.005972,0.024859,9.250354
1560355,93436,2024-08-31,-0.077390,214.110,6.840044e+08,3711.0,184996,0.105166,57.713252,59.310249,7.174588,59.301712,335.706250,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,0.221942,-0.100783,-0.006184,0.139169,20.343475
1560356,93436,2024-09-30,0.221942,261.630,8.390474e+08,3711.0,184996,0.105166,57.713252,72.473684,8.802612,72.758187,346.216280,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,-0.045025,-0.144312,0.014849,0.153502,20.547778
1560357,93436,2024-10-31,-0.045025,249.850,8.020335e+08,3711.0,184996,0.105166,57.713252,69.210526,8.406271,69.482219,369.690039,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,0.381469,0.302679,0.027542,0.140071,20.502661
1560358,93436,2024-11-30,0.381469,345.160,1.107984e+09,3711.0,184996,0.083346,53.762150,93.286486,11.394010,76.450592,306.628288,0.132692,0.084756,0.234822,0.183922,0.208738,0.190668,0.203448,0.099842,0.839369,0.274887,0.497103,0.690047,1.047903,1.798235,0.866390,25.589359,5.285830,1.475261,0.044858,0.0,0.0,-0.014637,23,0.170008,0.040696,0.043050,0.167882,20.825808
1560359,93436,2024-12-31,0.170008,403.840,1.298958e+09,3711.0,184996,0.083346,53.762150,109.145946,13.368497,89.698836,274.076290,0.132692,0.084756,0.234822,0.183922,0.208738,0.190668,0.203448,0.099842,0.839369,0.274887,0.497103,0.690047,1.047903,1.798235,0.866390,25.589359,5.285830,1.475261,0.044858,0.0,0.0,-0.014637,23,0.001882,0.389088,0.054301,0.171772,20.984828


Macro DataFrame shape: (408, 20)


,mthcaldt,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,1991-01-31,-0.001000,0.1,0.023836,-0.005920,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.000930,-0.50,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
1,1991-02-28,-0.001892,0.1,-0.041536,-0.004931,-0.003955,-0.029746,0.02,6.682109,0.001999,-0.093898,-0.007471,-0.40,-0.70,-0.3,-0.000014,26.9318,3.54,1.87,2.36
2,1991-03-31,-0.002384,0.2,0.091651,-0.007933,-0.009491,0.005330,0.02,6.872128,0.000704,0.100378,0.005295,-0.66,-0.61,0.5,-0.002975,22.1126,3.82,1.91,2.22
3,1991-04-30,-0.001568,0.2,0.024146,-0.009247,0.006162,-0.061953,0.00,6.825460,-0.000088,0.043726,-0.002343,-0.13,-0.08,0.2,-0.000742,19.0410,3.97,2.20,1.98
4,1991-05-31,-0.001709,-0.1,-0.082863,0.003497,0.011971,0.041812,0.00,6.908755,-0.002301,0.034535,-0.002986,-0.21,-0.34,0.6,0.002221,18.9272,4.03,2.39,1.90
403,2024-08-31,0.000718,0.2,0.006105,-0.006401,0.003767,0.094333,0.00,7.121252,-0.002100,-0.024829,-0.006629,0.00,0.03,-0.1,0.002110,14.4084,0.51,-0.95,1.59
404,2024-09-30,0.000895,-0.1,-0.030903,0.008110,0.001463,0.000459,-0.01,7.212294,0.003887,0.030865,0.003711,0.00,-0.19,0.4,0.000322,19.6750,0.27,-1.18,1.73
405,2024-10-31,0.001598,-0.1,-0.029656,-0.002832,0.003543,-0.007867,0.01,7.210818,-0.001050,-0.031354,0.002598,-0.20,-0.26,1.6,-0.000073,17.6597,0.29,-1.00,1.70
406,2024-11-30,0.000075,0.0,0.055299,-0.002629,0.001225,0.002425,0.00,7.178545,0.000319,0.023649,-0.000157,-0.30,-0.24,0.3,0.000641,19.9478,0.80,-0.41,1.53
407,2024-12-31,0.001426,0.1,-0.078988,-0.001472,0.002754,-0.010575,0.00,7.161622,0.002139,0.024533,-0.002535,-0.19,-0.09,0.8,0.000686,15.9822,1.13,-0.06,1.42


In [6]:
# =========================================================
# 1. Validate the cleaned micro table and normalize explicit features
# =========================================================
micro_normalize = 'rank'  # 'normal' or 'rank' for accounting-ratio features


micro_exclude_cols = {
    'permno', 'mthcaldt', 'mthcap', 'mthprc', 'siccd', 'gvkey', 'ffi49', 'target_ret_final'
}

micro_cols = [
    col for col in micro_df.columns
    if col not in micro_exclude_cols
]

micro_df = micro_df[['permno', 'mthcaldt', 'ffi49', 'target_ret_final'] + micro_cols].copy()
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)

if micro_normalize == 'rank':
    micro_norm = (
        micro_df.groupby('mthcaldt')[micro_cols].rank(pct=True) * 2.0 - 1.0
    )
elif micro_normalize == 'normal':
    monthly_mean = micro_df.groupby('mthcaldt')[micro_cols].transform('mean')
    monthly_std = micro_df.groupby('mthcaldt')[micro_cols].transform('std')
    micro_norm = (
        micro_df[micro_cols] - monthly_mean
    ) / (monthly_std + normalization_eps)
else:
    raise ValueError("micro_normalize must be 'normal' or 'rank'")
micro_df[micro_cols] = micro_norm.fillna(0.0).to_numpy()

micro_df = micro_df.sort_values(['mthcaldt', 'permno']).reset_index(drop=True)

print('Micro model panel:', micro_df.shape)

display(pd.concat([micro_df.head(3), micro_df.tail(3)]))

Micro model panel: (1560360, 37)


,permno,mthcaldt,ffi49,target_ret_final,mthret,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,MOM12m,mu_bench,sigma_bench,mthcap_log
0,10001,1991-01-31,31,0.012987,-0.328160,0.218879,-0.536902,-0.756098,-0.483054,-0.351283,0.527399,-0.256889,0.173899,-0.796009,-0.334812,0.159328,0.288565,-0.477352,0.663605,0.424137,-0.615458,0.554007,0.405765,0.273361,-0.590751,0.244853,0.569845,0.356984,0.514729,-0.292683,-0.281596,-0.168514,-0.401330,0.264492,-0.057966,-0.986696,-0.899905
1,10009,1991-01-31,0,0.038462,-0.586950,0.005702,-0.009186,0.000950,-0.004118,0.002217,0.032309,0.000317,-0.000950,-0.004118,-0.029458,-0.000634,0.008236,0.001584,0.002217,0.000317,-0.025974,0.035793,0.000317,-0.020272,0.038961,0.002534,0.004118,0.012987,0.002534,-0.292683,-0.281596,-0.168514,-0.002217,0.450111,0.172632,-0.559709,-0.961989
2,10010,1991-01-31,12,0.058824,0.854926,-0.882167,0.863161,0.937282,0.832753,0.958822,0.947418,0.275895,0.175166,0.735192,-0.380424,0.454545,0.683244,0.877732,-0.482420,-0.530567,0.683244,-0.536902,-0.608489,-0.318340,0.574913,0.712385,-0.457080,0.641432,0.228381,0.707317,-0.281596,-0.168514,0.730757,0.998733,0.999366,0.997466,-0.119417
1560357,93426,2024-12-31,37,-0.007243,0.721739,0.657739,-0.152696,0.424000,-0.563130,-0.094261,0.214609,-0.135304,-0.105391,0.113739,-0.149217,-0.259826,-0.040696,0.449043,-0.352348,-0.302261,0.661913,-0.088696,-0.609043,0.523478,0.627130,0.260522,0.102609,0.626435,-0.162783,0.556174,-0.448696,-0.208348,0.070609,-0.858087,-0.835826,-0.236870,-0.672348
1560358,93434,2024-12-31,1,0.147685,0.901913,0.995130,-0.805913,-0.713391,-0.841391,-0.707130,-0.549217,-0.819130,-0.791304,-0.583304,-0.873391,-0.842783,-0.868522,-0.402435,0.414261,-0.938087,0.954087,-0.746087,0.094957,-0.986087,-0.779478,-0.300174,-0.639652,-0.659826,-0.273043,0.592348,-0.448696,-0.208348,-0.811478,-0.923478,0.825391,0.975652,-0.988870
1560359,93436,2024-12-31,23,0.001882,0.918609,-0.859478,0.885913,0.924174,0.871304,0.941565,0.987130,0.499130,-0.148522,-0.588870,0.440696,0.682783,0.807304,0.071304,-0.421913,-0.414261,0.554783,-0.001739,-0.419130,0.495652,-0.076174,0.460174,0.831652,-0.286957,0.200696,0.512348,-0.448696,-0.208348,0.433739,0.451826,0.764870,0.641739,0.997217


In [7]:
# =========================================================
# 2. Macro：初始训练期统一拟合，之后按时间块更新
# =========================================================
def standardize_macro(
    macro_data,
    macro_cols,
    model_dates,
    initial_months,
    update_months,
):
    macro = macro_data.set_index('mthcaldt').loc[model_dates, macro_cols].copy()
    macro_z = pd.DataFrame(index=model_dates, columns=macro_cols, dtype=float)

    initial_dates = model_dates[:initial_months]
    initial_mean = macro.loc[initial_dates].mean()
    initial_std = macro.loc[initial_dates].std(ddof=0)
    macro_z.loc[initial_dates] = (
        (macro.loc[initial_dates] - initial_mean) /
        (initial_std + normalization_eps)
    ).to_numpy()

    for start in range(initial_months, len(model_dates), update_months):
        # print(f'>>> Standardizing Macro features for block starting at {model_dates[start]}...')
        block_dates = model_dates[start:start + update_months]
        history = macro.iloc[:start]
        history_mean = history.mean()
        history_std = history.std(ddof=0)
        macro_z.loc[block_dates] = (
            (macro.loc[block_dates] - history_mean) /
            (history_std + normalization_eps)
        ).to_numpy()

    macro_z.index.name = 'mthcaldt'
    return macro_z.reset_index()

macro_cols = [
    col for col in macro_df.columns
    if col != 'mthcaldt'
]

macro_df['mthcaldt'] = pd.to_datetime(macro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df = macro_df.sort_values('mthcaldt').reset_index(drop=True)

model_dates = pd.DatetimeIndex(np.sort(micro_df['mthcaldt'].unique()))

macro_df = standardize_macro(
    macro_data=macro_df,
    macro_cols=macro_cols,
    model_dates=model_dates,
    initial_months=initial_train_months,
    update_months=test_months,
)
pd.concat([macro_df.head(5), macro_df.tail(5)])

,mthcaldt,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,1991-01-31,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
1,1991-02-28,-2.192501,0.729992,-0.817965,-1.469064,-1.649264,-1.007572,1.775543,-3.404600,0.550763,-1.445587,-1.227084,-1.945786,-3.240398,-0.599735,0.013918,0.924305,-0.184668,-0.058573,0.470548
2,1991-03-31,-2.549766,1.417942,1.947638,-2.076241,-3.018345,0.041438,1.775543,-2.399603,0.198073,1.546106,0.850978,-3.281213,-2.811241,0.849289,-1.146876,0.257253,0.008029,-0.022803,0.211304
3,1991-04-30,-1.957019,1.417942,0.545911,-2.342198,0.852805,-1.970789,0.173549,-2.646426,-0.017617,0.673709,-0.392313,-0.558996,-0.283985,0.305905,-0.271394,-0.167903,0.111260,0.236528,-0.233113
4,1991-05-31,-2.059530,-0.645909,-1.676114,0.236066,2.289584,1.132493,0.173549,-2.205888,-0.620199,0.532180,-0.496941,-0.969897,-1.523771,1.030417,0.890301,-0.183655,0.152552,0.406434,-0.381253
403,2024-08-31,-0.019744,0.360853,0.054717,-0.858590,0.157267,2.364269,0.067085,-0.036957,-0.404763,-0.034278,-0.618382,0.033549,0.170575,-0.106644,0.687854,-0.684488,-2.068622,-2.216100,-1.064560
404,2024-09-30,0.002850,-0.163222,-0.189031,0.681921,-0.066753,-0.049609,-0.415929,0.225697,0.730580,0.042472,0.346484,0.033549,-0.915719,0.336030,0.108108,-0.033047,-2.216690,-2.410739,-0.868202
405,2024-10-31,0.092972,-0.163222,-0.180820,-0.479714,0.135471,-0.263698,0.550100,0.221439,-0.205664,-0.043270,0.242691,-1.037861,-1.261358,1.398448,-0.020236,-0.282325,-2.204351,-2.258413,-0.910279
406,2024-11-30,-0.102179,0.011470,0.378732,-0.458106,-0.089945,0.000934,0.067085,0.128332,0.053851,0.032528,-0.014406,-1.573566,-1.162604,0.247495,0.211272,0.000696,-1.889707,-1.759122,-1.148713
407,2024-12-31,0.070973,0.186161,-0.505745,-0.335332,0.058751,-0.333345,0.067085,0.079508,0.399053,0.033746,-0.236290,-0.984291,-0.421949,0.690169,0.226152,-0.489820,-1.686113,-1.462933,-1.302994


In [8]:
# =========================================================
# 3. 最后合并标准化后的 Micro 和 Macro
# =========================================================
processed_df = (
    micro_df
    .merge(macro_df, on='mthcaldt', how='left', validate='many_to_one')
    .sort_values(['mthcaldt', 'permno'])
    .reset_index(drop=True)
)

print(f'Data preprocessing complete. Final shape: {processed_df.shape}')
print(f'Micro features: {len(micro_cols)}; Macro features: {len(macro_cols)}')

pd.concat([processed_df.head(5), processed_df.tail(5)])

Data preprocessing complete. Final shape: (1560360, 56)
Micro features: 33; Macro features: 19


,permno,mthcaldt,ffi49,target_ret_final,mthret,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,MOM12m,mu_bench,sigma_bench,mthcap_log,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,10001,1991-01-31,31,0.012987,-0.328160,0.218879,-0.536902,-0.756098,-0.483054,-0.351283,0.527399,-0.256889,0.173899,-0.796009,-0.334812,0.159328,0.288565,-0.477352,0.663605,0.424137,-0.615458,0.554007,0.405765,0.273361,-0.590751,0.244853,0.569845,0.356984,0.514729,-0.292683,-0.281596,-0.168514,-0.401330,0.264492,-0.057966,-0.986696,-0.899905,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
1,10009,1991-01-31,0,0.038462,-0.586950,0.005702,-0.009186,0.000950,-0.004118,0.002217,0.032309,0.000317,-0.000950,-0.004118,-0.029458,-0.000634,0.008236,0.001584,0.002217,0.000317,-0.025974,0.035793,0.000317,-0.020272,0.038961,0.002534,0.004118,0.012987,0.002534,-0.292683,-0.281596,-0.168514,-0.002217,0.450111,0.172632,-0.559709,-0.961989,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
2,10010,1991-01-31,12,0.058824,0.854926,-0.882167,0.863161,0.937282,0.832753,0.958822,0.947418,0.275895,0.175166,0.735192,-0.380424,0.454545,0.683244,0.877732,-0.482420,-0.530567,0.683244,-0.536902,-0.608489,-0.318340,0.574913,0.712385,-0.457080,0.641432,0.228381,0.707317,-0.281596,-0.168514,0.730757,0.998733,0.999366,0.997466,-0.119417,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
3,10016,1991-01-31,34,0.161039,-0.695280,0.594552,-0.727589,-0.723155,-0.853659,-0.960089,-0.486855,-0.371555,-0.181501,-0.333545,-0.486221,0.375356,0.482420,0.291099,0.669940,0.422236,-0.335445,-0.361419,0.528033,0.321508,-0.173266,0.365220,-0.312639,0.501425,0.683877,0.552106,-0.281596,-0.168514,0.820082,-0.766867,-0.865695,0.092176,0.185303,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
4,10019,1991-01-31,38,0.076923,-0.436490,0.272094,-0.408299,-0.658537,-0.187203,-0.337346,0.242952,0.785873,0.632563,0.500792,0.547038,0.741527,-0.789357,0.645233,-0.551473,-0.642065,-0.696547,0.901806,-0.809946,0.814381,0.913209,0.413367,-0.290466,0.915109,-0.427304,0.863161,-0.281596,-0.168514,0.371555,-0.870130,-0.909408,0.255622,-0.519797,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
1560355,93374,2024-12-31,46,0.012492,-0.394783,0.446261,0.550609,0.870609,-0.490783,0.152696,0.385739,-0.332870,-0.411478,-0.879652,-0.389217,-0.362783,-0.074087,-0.725217,-0.261913,0.594435,0.619478,-0.571478,0.316174,-0.132522,-0.400348,-0.233391,-0.330783,0.168000,-0.013565,-0.384696,-0.448696,-0.208348,0.250783,-0.138087,-0.307826,-0.483130,0.446957,0.070973,0.186161,-0.505745,-0.335332,0.058751,-0.333345,0.067085,0.079508,0.399053,0.033746,-0.236290,-0.984291,-0.421949,0.690169,0.226152,-0.489820,-1.686113,-1.462933,-1.302994
1560356,93397,2024-12-31,1,-0.057236,-0.536696,-0.051826,0.965913,0.844174,0.072696,0.650087,0.812174,-0.146435,-0.641739,-0.757217,-0.211826,-0.276522,-0.093565,-0.327304,-0.288000,0.873391,-0.273391,0.259130,-0.549913,-0.842783,-0.891478,0.186783,0.586087,0.252870,-0.149913,-0.384696,-0.448696,-0.208348,0.161043,0.399652,0.166609,-0.481043,-0.535304,0.070973,0.186161,-0.5057

In [ ]:
class LinearQuantile(nn.Module):
    def __init__(self, input_dim, quantile_levels):
        super().__init__()
        self.linear = nn.Linear(input_dim, len(quantile_levels))
        self.register_buffer("quantile_levels", torch.tensor(quantile_levels, dtype=torch.float32).view(1, -1))

    def forward(self, x):
        return self.linear(x)

In [ ]:
def weighted_pinball_loss(pred, target, weight, quantile_levels):
    error = target - pred
    loss = torch.maximum(quantile_levels * error, (quantile_levels - 1) * error)
    return (loss.mean(dim=1) * weight).sum() / weight.sum().clamp_min(1e-12)


In [ ]:
def rolling_window_generator(panel):
    dates = np.sort(panel["mthcaldt"].unique())

    current_split_idx = initial_train_months

    while current_split_idx + val_months < len(dates):
        start_idx = current_split_idx - initial_train_months

        train_dates = dates[start_idx:current_split_idx]
        val_dates = dates[current_split_idx:current_split_idx + val_months]
        test_dates = dates[current_split_idx + val_months:current_split_idx + val_months + test_months]

        train_df = panel[panel["mthcaldt"].isin(train_dates)].copy()
        val_df = panel[panel["mthcaldt"].isin(val_dates)].copy()
        test_df = panel[panel["mthcaldt"].isin(test_dates)].copy()

        yield train_df, val_df, test_df

        current_split_idx += test_months

In [ ]:
# ==========================================
# 训练单个 Linear Quantile
# ==========================================
def train_linear_quantile(train_df, val_df):
    feature_mean, feature_std = fit_feature_scaler(train_df)

    X_train = transform_features(train_df, feature_mean, feature_std)
    X_val = transform_features(val_df, feature_mean, feature_std)

    y_train = train_df["target_ret_final"].to_numpy(dtype=np.float32)
    y_val = val_df["target_ret_final"].to_numpy(dtype=np.float32)

    w_train = add_month_equal_weights(train_df).astype(np.float32)
    w_val = add_month_equal_weights(val_df).astype(np.float32)

    train_dataset = TensorDataset(
        torch.tensor(X_train), torch.tensor(y_train), torch.tensor(w_train)
    )
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    X_val = torch.tensor(X_val, device=device)
    y_val = torch.tensor(y_val, device=device).unsqueeze(1)
    w_val = torch.tensor(w_val, device=device)
    quantiles = torch.tensor(QUANTILE_LEVELS, dtype=torch.float32, device=device).view(1, -1)

    model = LinearQuantile(len(FEATURE_COLS), QUANTILE_LEVELS).to(device)

    # 用训练样本的无条件分位数初始化截距
    initial_quantiles = np.quantile(y_train, QUANTILE_LEVELS)
    with torch.no_grad():
        model.linear.weight.zero_()
        model.linear.bias.copy_(torch.tensor(initial_quantiles, dtype=torch.float32, device=device))

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0

    for epoch in range(MAX_EPOCHS):
        model.train()

        for batch_x, batch_y, batch_w in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device).unsqueeze(1)
            batch_w = batch_w.to(device)

            optimizer.zero_grad()
            pred = model(batch_x)

            pinball = weighted_pinball_loss(pred, batch_y, batch_w, quantiles)
            crossing = torch.relu(pred[:, :-1] - pred[:, 1:]).mean()
            loss = pinball + 0.10 * crossing

            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = weighted_pinball_loss(val_pred, y_val, w_val, quantiles).item()

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                break

    model.load_state_dict(best_state)
    return model, feature_mean, feature_std, best_val_loss, epoch + 1


# ==========================================
# 根据分位数函数计算均值、波动率、VaR 和 ES
# ==========================================
def add_quantile_distribution_results(output, quantile_pred):
    levels = QUANTILE_LEVELS
    quantile_pred = np.sort(quantile_pred, axis=1)  # 防止分位数交叉
    trapz = getattr(np, "trapezoid", np.trapz)

    full_levels = np.r_[0.0, levels, 1.0]
    full_quantiles = np.column_stack((quantile_pred[:, 0], quantile_pred, quantile_pred[:, -1]))

    pred_mean = trapz(full_quantiles, full_levels, axis=1)
    second_moment = trapz(full_quantiles ** 2, full_levels, axis=1)

    output["pred_mean"] = pred_mean
    output["pred_vol"] = np.sqrt(np.maximum(second_moment - pred_mean ** 2, 0.0))

    for alpha in ALPHAS:
        suffix = ALPHA_SUFFIX[alpha]
        alpha_idx = np.flatnonzero(np.isclose(levels, alpha))[0]
        below_alpha = levels < alpha

        var_value = quantile_pred[:, alpha_idx]
        tail_levels = np.r_[0.0, levels[below_alpha], alpha]
        tail_quantiles = np.column_stack((quantile_pred[:, 0], quantile_pred[:, below_alpha], var_value))

        output[f"var_{suffix}"] = var_value
        output[f"es_{suffix}"] = -trapz(tail_quantiles, tail_levels, axis=1) / alpha

    output["quantile_levels"] = json.dumps(levels.tolist())
    output["quantile_vec"] = [json.dumps(row.tolist()) for row in quantile_pred]

    return output


# ==========================================
# 完整滚动预测
# ==========================================
def run_linear_quantile(panel):
    results = []

    for train_df, val_df, test_df in rolling_window_generator(panel):
        train_start = train_df["mthcaldt"].min().strftime("%Y-%m")
        test_start = test_df["mthcaldt"].min().strftime("%Y-%m")
        print(f"Train starts {train_start} | Test starts {test_start}")

        model, feature_mean, feature_std, val_loss, epochs_run = train_linear_quantile(train_df, val_df)

        test_df = test_df.sort_values(["mthcaldt", "permno"]).copy()
        X_test = transform_features(test_df, feature_mean, feature_std)
        X_test = torch.tensor(X_test, device=device)

        model.eval()
        with torch.no_grad():
            quantile_pred = model(X_test).cpu().numpy()

        output = test_df[["permno", "mthcaldt", "target_ret_final"]].copy()
        output = output.rename(columns={"mthcaldt": "forecast_date", "target_ret_final": "realized_ret"})
        output["target_date"] = output["forecast_date"] + pd.offsets.MonthEnd(1)
        output["model_id"] = "linear_quantile"
        output["validation_loss"] = val_loss
        output["epochs_run"] = epochs_run

        results.append(add_quantile_distribution_results(output, quantile_pred))

    return pd.concat(results, ignore_index=True)


# ==========================================
# 运行
# ==========================================

linear_quantile_forecasts = run_linear_quantile(processed_df)
display(linear_quantile_forecasts.head())

In [8]:
# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col
        
    def expanding_window_generator(self, initial_train_months=180, val_months=12, test_months=12):
        dates = np.sort(self.df[self.date_col].unique())
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months  < len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months
            
    # （固定 15 年滚动窗口版）
    def rolling_window_generator(self, train_months=180, val_months=12, test_months=12):
        """
        固定训练窗口长度的滚动窗口生成器。
        参数：
        train_months: 固定训练集包含的月数。默认 180 个月(15年)。
        """
        # 确保日期是有序的
        dates = np.sort(self.df[self.date_col].unique())
        
        # 初始时，当前切分点就是训练集的总长度（180个月）
        current_split_idx = train_months
        
        # 只要后面还有足够的月份可以分给验证集和测试集，就继续滚动
        while current_split_idx + val_months < len(dates):
            
            # 【核心修改点】让训练集的起点跟着一起滚动，确保长度永远等于 train_months
            start_idx = current_split_idx - train_months
            
            # 切分对应的日期数组
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            # 根据日期过滤出对应的数据集
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            
            # 向前推進一个测试窗口的长度
            current_split_idx += test_months

In [ ]:
# ==========================================
# Fused MDN
# Macro、Micro、Industry Embedding 进入同一个网络
# num_components 可以设置为 1 或 3
# ==========================================
class FusedMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, ind_emb_dim=6,
        hidden_dim=64, num_components=3, dropout_rate=0.1, dist_type="normal",):
        super().__init__()

        self.num_components = num_components
        self.dist_type = dist_type

        # ------------------------------------------
        # 1. 行业 embedding
        # ------------------------------------------
        self.ind_embedding = nn.Embedding(num_embeddings=num_industries, embedding_dim=ind_emb_dim,)

        # Macro、Micro 和行业 embedding 全部拼接
        total_input_dim = macro_dim + micro_dim + ind_emb_dim

        target_dim = 8

        # ------------------------------------------
        # 2. 统一的 Fused Backbone
        # ------------------------------------------
        self.fused_extractor = nn.Sequential(
            nn.Linear(total_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim // 2, target_dim),
            nn.LayerNorm(target_dim),
            nn.ELU(),
        )

        # ------------------------------------------
        # 3. Distribution Heads
        # ------------------------------------------

        # K=1 时 pi 必然等于 1，所以不建立无效的 pi_head
        if num_components > 1:
            self.pi_head = nn.Linear(
                target_dim,
                num_components,
            )
        else:
            self.pi_head = None

        self.mu_head = nn.Linear(
            target_dim,
            num_components,
        )

        self.sigma_head = nn.Linear(
            target_dim,
            num_components,
        )

        if dist_type == "t":
            self.nu_head = nn.Linear(
                target_dim,
                num_components,
            )
        else:
            self.nu_head = None

        # ------------------------------------------
        # 4. 初始化
        # ------------------------------------------
        nn.init.xavier_normal_(
            self.mu_head.weight,
            gain=0.1,
        )
        nn.init.zeros_(self.mu_head.bias)

        # 初始 component scale 约为 9%–10%
        nn.init.constant_(
            self.sigma_head.bias,
            -2.35,
        )

        if self.pi_head is not None:
            # 初始 mixture weights 大致对称
            nn.init.zeros_(self.pi_head.bias)

        if self.nu_head is not None:
            # softplus(5) + 2.01 约等于 7
            nn.init.constant_(
                self.nu_head.bias,
                5.0,
            )

    def forward(
        self,
        x_macro,
        x_micro,
        x_micro_ind,
    ):
        # ------------------------------------------
        # 1. 行业 embedding
        # ------------------------------------------
        industry_embedding = self.ind_embedding(x_micro_ind)

        # ------------------------------------------
        # 2. 拼接全部条件信息
        # ------------------------------------------
        x_fused = torch.cat(
            [
                x_macro,
                x_micro,
                industry_embedding,
            ],
            dim=-1,
        )

        # ------------------------------------------
        # 3. 统一 backbone
        # ------------------------------------------
        h_fused = self.fused_extractor(x_fused)

        # ------------------------------------------
        # 4. Mixture weights
        # ------------------------------------------
        if self.pi_head is None:
            # K=1：softmax([0]) = [1]
            pi_logits = h_fused.new_zeros(
                (h_fused.shape[0], 1)
            )
        else:
            pi_logits = self.pi_head(h_fused)

        # ------------------------------------------
        # 5. Component parameters
        # ------------------------------------------
        mu = self.mu_head(h_fused)

        sigma = (
            F.softplus(self.sigma_head(h_fused))
            + sigma_floor
        )

        nu = None
        if self.nu_head is not None:
            nu = (
                F.softplus(self.nu_head(h_fused))
                + 2.01
            )

        return pi_logits, mu, sigma, nu

In [9]:
# ==========================================
# 简化版 Structured MDN：不使用 FiLM
# ==========================================
class StructuredMDN_NoFiLM(nn.Module):
    def __init__(
        self,
        macro_dim,
        micro_dim,
        num_industries,
        ind_emb_dim=6,
        micro_hidden_dim=64,
        macro_hidden_dim=32,
        num_components=3,
        dropout_rate=0.1,
        dist_type='normal',
    ):
        super().__init__()

        self.num_components = num_components
        self.dist_type = dist_type

        # ------------------------------------------
        # 1. 行业 embedding
        # ------------------------------------------
        self.ind_embedding = nn.Embedding(
            num_embeddings=num_industries,
            embedding_dim=ind_emb_dim,
        )

        total_micro_dim = micro_dim + ind_emb_dim

        # ------------------------------------------
        # 2. Macro 网络：只预测 mixture weights
        # ------------------------------------------
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, macro_hidden_dim),
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(macro_hidden_dim, macro_hidden_dim // 2),
            nn.LayerNorm(macro_hidden_dim // 2),
            nn.ELU(),
        )

        self.pi_head = nn.Linear(
            macro_hidden_dim // 2,
            num_components,
        )

        # ------------------------------------------
        # 3. Micro 网络：预测各 mixture component
        # ------------------------------------------
        target_dim = 8

        self.micro_extractor = nn.Sequential(
            nn.Linear(total_micro_dim, micro_hidden_dim),
            nn.LayerNorm(micro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(micro_hidden_dim, micro_hidden_dim // 2),
            nn.LayerNorm(micro_hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(micro_hidden_dim // 2, target_dim),
            nn.LayerNorm(target_dim),
            nn.ELU(),
        )

        self.mu_head = nn.Linear(target_dim, num_components)
        self.sigma_head = nn.Linear(target_dim, num_components)

        if self.dist_type == 't':
            self.nu_head = nn.Linear(target_dim, num_components)
        else:
            self.nu_head = None

        # ------------------------------------------
        # 4. 初始化
        # ------------------------------------------
        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        nn.init.zeros_(self.mu_head.bias)

        # 初始 component scale 约为 9%–10%
        nn.init.constant_(self.sigma_head.bias, -2.35)

        if self.nu_head is not None:
            # softplus(5) + 2.01 ≈ 7
            nn.init.constant_(self.nu_head.bias, 5.0)

    def forward(self, x_macro, x_micro, x_micro_ind):
        # Macro information
        h_macro = self.macro_net(x_macro)
        pi_logits = self.pi_head(h_macro)

        # Micro information + industry embedding
        industry_embedding = self.ind_embedding(x_micro_ind)

        x_micro_combined = torch.cat(
            [x_micro, industry_embedding],
            dim=-1,
        )

        h_micro = self.micro_extractor(x_micro_combined)

        # 不再进行 FiLM 调制
        mu = self.mu_head(h_micro)

        sigma = (
            F.softplus(self.sigma_head(h_micro))
            + sigma_floor
        )

        nu = None
        if self.nu_head is not None:
            nu = F.softplus(self.nu_head(h_micro)) + 2.01

        return pi_logits, mu, sigma, nu

## Structured MDN with FiLM

The executable definition below is the production architecture. Obsolete experimental model definitions were removed to keep top-to-bottom execution unambiguous.


In [10]:
# ==========================================
# 2. 模型定义：结构化 MDN (Mixture Density Network) + FiLM 调制
# ==========================================
class StructuredMDN_with_FiLM(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, ind_emb_dim=6, 
                 micro_hidden_dim=64, macro_hidden_dim=32, num_components=3, 
                 dropout_rate=0.1, dist_type='normal'):
        """
        结构化 MDN (加入 FiLM 调制机制)
        - 保持 2D 截面输入，不引入序列记忆 (无 GRU)
        - 宏观特征不仅预测 Regime 概率，还直接调控微观特征的表达
        """
        super(StructuredMDN_with_FiLM, self).__init__()
        self.num_components = num_components
        self.dist_type = dist_type

        # ++++++++++++++++++++++++++++++++++++++++++
        # 新增：行业实体嵌入层 (Entity Embedding)
        # ++++++++++++++++++++++++++++++++++++++++++
        self.ind_embedding = nn.Embedding(num_embeddings=num_industries, embedding_dim=ind_emb_dim)
        
        # 计算拼接后的微观特征总维度 = 连续特征维度 + 行业嵌入维度(6)
        total_micro_dim = micro_dim + ind_emb_dim
        # ==========================================
        # 通道 A: Macro Network (标准 MLP)
        # 输入: (Batch, Macro_Dim) 截面宏观数据
        # ==========================================
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, macro_hidden_dim),
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(macro_hidden_dim , macro_hidden_dim//2), # 🌟 优化：增加一层，让宏观特征更丰富
            nn.LayerNorm(macro_hidden_dim//2),
            nn.ELU()
        )
        
        # 门控预测头：基于宏观隐状态预测机制概率 pi
        self.pi_head = nn.Linear(macro_hidden_dim//2, num_components)
        
        # ==========================================
        # ★ FiLM 调制发生器 (Modulation Generators) ★
        # ==========================================
        # 目标是对齐微观网络的最后一层输出维度
        target_dim = 8
        
        # Gamma 负责“缩放(Scale)”微观特征，Beta 负责“平移(Shift)”微观特征
        self.film_gamma = nn.Linear(macro_hidden_dim//2, target_dim)
        self.film_beta = nn.Linear(macro_hidden_dim//2, target_dim)
        
        # ==========================================
        # 通道 B: Micro Network (专家网络)
        # 输入: (Batch, Micro_Dim) 截面微观数据
        # ==========================================
        self.micro_extractor = nn.Sequential(
            nn.Linear(total_micro_dim, micro_hidden_dim),
            nn.LayerNorm(micro_hidden_dim), # 使用 LayerNorm 防止截面前视偏差
            nn.ELU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(micro_hidden_dim, micro_hidden_dim // 2),
            nn.LayerNorm(micro_hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(micro_hidden_dim // 2, target_dim),
            nn.LayerNorm(target_dim),
            nn.ELU()
        )
        
        # 预测头 (Heads): 注意，这里输入的是被 FiLM 调制后的特征
        self.mu_head = nn.Linear(target_dim, num_components)
        self.sigma_head = nn.Linear(target_dim, num_components)
        # 🌟 仅在选择了 t 分布时，才挂载自由度预测头
        if self.dist_type == 't':
            self.nu_head = nn.Linear(target_dim, num_components)    

        # ==========================================
        # 🌟 优化：MDN 安全初始化策略
        # ==========================================
        # 1. 均值头：初始化为极小的值，防止初始预期收益率过高
        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        
        # 2. Sigma head starts near 9% monthly scale before learning.
        nn.init.constant_(self.sigma_head.bias, -2.35)
        
        if self.dist_type == 't':
            # 自由度头：初始化 nu 较大（例如 10），使其初始状态接近正态分布
            # 因为你的代码里有 + 2.01 的截断，softplus(x) + 2.01 = 10 -> x ≈ 2.07
            # nn.init.constant_(self.nu_head.bias, 2.07)  
            nn.init.constant_(self.nu_head.bias, 5)

    def forward(self, x_macro, x_micro, x_micro_ind):
        """
        x_macro: (Batch, Macro_Dim) - 当前时刻的宏观因子 (2D张量)
        x_micro: (Batch, Micro_Dim) - 当前时刻的微观因子 (2D张量)
        x_micro_ind: (Batch,) - 行业整数索引 0~47 (Long Tensor)
        """
        # ------------------------------------------
        # 1. 宏观特征提取
        # ------------------------------------------
        h_macro = self.macro_net(x_macro) # 形状: (Batch, macro_hidden_dim)
        
        # 预测门控权重
        pi_logits = self.pi_head(h_macro)
        
        # ------------------------------------------
        # 2. 生成 FiLM 参数 (根据当前宏观环境)
        # ------------------------------------------
        gamma = self.film_gamma(h_macro) # 形状: (Batch, target_dim)
        beta = self.film_beta(h_macro)   # 形状: (Batch, target_dim)
        
        # ++++++++++++++++++++++++++++++++++++++++++
        # 3. 新增：Embedding 降维与特征拼接
        # ++++++++++++++++++++++++++++++++++++++++++
        # 将 [0, 1, ..., 49] 的整数索引，转化为 6 维稠密向量
        ind_emb = self.ind_embedding(x_micro_ind) # 形状: (Batch, 6)
        
        # 在特征维度 (dim=-1) 上拼接连续特征和行业 Embedding
        x_micro_combined = torch.cat([x_micro, ind_emb], dim=-1)
        
        # 修改：将拼接后的组合特征送入提取器
        h_micro = self.micro_extractor(x_micro_combined)
        
        # ------------------------------------------
        # 4. ★ 核心：执行 FiLM 调制 ★
        # ------------------------------------------
        # 宏观环境直接介入微观特征的表达。
        h_modulated = (1.0 + gamma) * h_micro + beta
        
        # ------------------------------------------
        # 5. 专家网络最终预测
        # ------------------------------------------
        # Mu 路径：直接使用纯粹的、未受宏观污染的微观特征
        mu = self.mu_head(h_micro)
        sigma = nn.functional.softplus(self.sigma_head(h_modulated)) + sigma_floor
        
        # 3. 🌟 根据开关决定 Nu
        nu = None # 默认为空
        if self.dist_type == 't':
            # 保证自由度严格大于 2，以确保方差存在
            nu = nn.functional.softplus(self.nu_head(h_modulated)) + 2.01
            
        # 统一返回4个值，如果是 normal，nu 就是 None
        return pi_logits, mu, sigma, nu


## Weighted likelihood and training

Training uses explicit month-aware sample weights and device-side batch indexing for Kaggle/Colab GPU runs.


In [11]:

# ==========================================
# 4. 稳健的 NLL 损失函数 (支持时间衰减权重)
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, target, weights, dist_type='normal'):
    """
    weights: shape (batch_size, 1)，距离现在越近权重大
    """
    log_pi = torch.log_softmax(pi_logits, dim=-1)

# 🌟 核心开关：底层分布切换
    if dist_type == 't':
        dist = torch.distributions.StudentT(df=nu, loc=mu, scale=sigma)
    else:
        dist = torch.distributions.Normal(mu, sigma)
    
    # 分布的 log_prob
    log_dist = dist.log_prob(target.expand_as(mu))
    log_mix = log_pi + log_dist
    
    # 每个样本基础的 NLL loss
    loss = -torch.logsumexp(log_mix, dim=-1) 
    
    # 乘以时间衰减权重后求平均
    # 确保 weights 的 shape 与 loss 对齐
    weighted_loss = loss * weights.squeeze()
    
    return weighted_loss.sum()/ weights.sum()

In [12]:
# ==========================================
# 5. 极速版双通道训练循环 (All-in-VRAM 加速)
# ==========================================
def tensor_batch_indices(length, batch_size, device, shuffle=True):
    if shuffle:
        indices = torch.randperm(length, device=device)
    else:
        indices = torch.arange(length, device=device)
    for start in range(0, length, batch_size):
        yield indices[start:start + batch_size] 



def train_structured_mdn(model, 
                         X_train_macro, X_train_micro, X_train_industry, y_train, w_train, 
                         X_val_macro=None, X_val_micro=None, X_val_industry=None, y_val=None, w_val=None,
                         epochs=100, batch_size=1024, lr=1e-3, patience=25): 
    if run_test == "quick":
        epochs = 2
        batch_size = 512
        patience = 1

    # 2. 🌟 智能判定：只有当验证标签不为空时，才实例化验证集 Loader
    has_validation = (y_val is not None)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.8, patience=8, min_lr=1e-6,
    )

    # 保持 target 和 weight 的二维形状
    train_y = y_train.unsqueeze(1)
    train_w = w_train.unsqueeze(1)
    train_size = len(y_train)

    # tr_dataset = TensorDataset(X_train_macro, X_train_micro, X_train_industry, y_train.unsqueeze(1), w_train.unsqueeze(1))
    # train_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)

    train_loss_history = []

    # 3. 🌟 智能判定：只有当验证标签不为空时，才实例化验证集 Loader
    if has_validation:
        # va_dataset = TensorDataset(X_val_macro, X_val_micro, X_val_industry, y_val.unsqueeze(1), w_val.unsqueeze(1))
        # val_loader = DataLoader(va_dataset, batch_size=batch_size, shuffle=False)

        val_y = y_val.unsqueeze(1)
        val_w = w_val.unsqueeze(1)
        val_size = len(y_val)
        
        epochs_no_improve = 0
        best_model_state = None
    best_val_loss = float('inf')
    val_loss_history = []
    
    
    for epoch in range(epochs):
        model.train()
        train_loss_numerator = 0.0
        train_weight_sum = 0.0


        for batch_idx in tensor_batch_indices(
                    train_size,
                    batch_size,
                    X_train_macro.device,
                    shuffle=True
                ):
            b_macro = X_train_macro[batch_idx]
            b_micro = X_train_micro[batch_idx]
            b_industry = X_train_industry[batch_idx]
            b_y = train_y[batch_idx]
            b_w = train_w[batch_idx]


        # for b_macro, b_micro, b_industry, b_y, b_w in train_loader:
            optimizer.zero_grad()
            
            pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
            optimizer.step()
            batch_weight = float(b_w.sum().item())
            train_loss_numerator += loss.item() * batch_weight
            train_weight_sum += batch_weight
            
        train_loss = train_loss_numerator / max(train_weight_sum, 1e-12)
        train_loss_history.append(train_loss)
        
        # Validation
        if has_validation:
            model.eval()
            val_loss_numerator = 0.0
            val_weight_sum = 0.0
            with torch.no_grad():
                for batch_idx in tensor_batch_indices(
                    val_size,
                    batch_size,
                    X_val_macro.device,
                    shuffle=False
                ):
                    b_macro = X_val_macro[batch_idx]
                    b_micro = X_val_micro[batch_idx]
                    b_industry = X_val_industry[batch_idx]
                    b_y = val_y[batch_idx]
                    b_w = val_w[batch_idx]

                    
                # for b_macro, b_micro, b_industry, b_y, b_w in val_loader:
                    pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
                    loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)
                    batch_weight = float(b_w.sum().item())
                    val_loss_numerator += loss.item() * batch_weight
                    val_weight_sum += batch_weight
            val_loss = val_loss_numerator / max(val_weight_sum, 1e-12)
            val_loss_history.append(val_loss)

            scheduler.step(val_loss)
                
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    break
    if has_validation and best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model, best_val_loss, train_loss_history, val_loss_history

In [13]:
# ==========================================
# 6. 固定 epoch 的 warm-start 训练：第一期 100 个 epoch，后续窗口 50 个 epoch
# ==========================================
def train_structured_mdn_fixed_epochs(model, 
                         X_train_macro, X_train_micro, X_train_industry, y_train, w_train, 
                         X_val_macro=None, X_val_micro=None, X_val_industry=None, y_val=None, w_val=None,
                         first_window_epochs=100, later_window_epochs=50,
                         batch_size=1024, lr=1e-3, patience=25, is_first_window=False):

    # ==========================================
    # 每个窗口都在传入模型的现有参数上继续训练。
    # Train + Val 合并训练；第一轮 100 epoch，后续窗口 50 epoch。
    # ==========================================
    if run_test == 'quick':
        epochs_to_run = 2 if is_first_window else 1
    else:
        epochs_to_run = first_window_epochs if is_first_window else later_window_epochs

    # 合并 Train 和 Validation
    combined_macro = torch.cat([X_train_macro, X_val_macro], dim=0)
    combined_micro = torch.cat([X_train_micro, X_val_micro], dim=0)
    combined_industry = torch.cat([X_train_industry, X_val_industry], dim=0)
    combined_y = torch.cat([y_train, y_val], dim=0).unsqueeze(1)
    combined_w = torch.cat([w_train, w_val], dim=0).unsqueeze(1)

    # -----------------------------------------
    # 使用 TensorDataset 和 DataLoader 进行批处理
    # combined_dataset = TensorDataset(
    #     combined_macro,
    #     combined_micro,
    #     combined_industry,
    #     combined_y,
    #     combined_w,
    # )
    # train_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=True)
    # -----------------------------------------

    combined_size = combined_y.shape[0]

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.8, patience=8, min_lr=1e-6,
    )
    
    # 🌟 新增：用于记录 Loss 历史
    train_loss_history = []
    val_loss_history = []
    
    for epoch in range(epochs_to_run):
        model.train()
        train_loss_numerator = 0.0
        train_weight_sum = 0.0

        # -----------------------------------------
        # 使用自定义的 tensor_batch_indices 进行批处理
        for batch_idx in tensor_batch_indices(combined_size, batch_size, combined_macro.device, shuffle=True):
            b_macro = combined_macro[batch_idx]
            b_micro = combined_micro[batch_idx]
            b_industry = combined_industry[batch_idx]
            b_y = combined_y[batch_idx]
            b_w = combined_w[batch_idx] 
        # -----------------------------------------

        # -----------------------------------------
        # 使用 TensorDataset 和 DataLoader 进行批处理
        # for b_macro, b_micro, b_industry, b_y, b_w in train_loader:
        # -----------------------------------------


            optimizer.zero_grad()

            pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)

            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            batch_weight = float(b_w.sum().item())
            train_loss_numerator += loss.item() * batch_weight
            train_weight_sum += batch_weight
            
        train_loss = train_loss_numerator / max(train_weight_sum, 1e-12)
        train_loss_history.append(train_loss)
        val_loss_history.append(train_loss)   # 无独立 val，用 train_loss 占位
        scheduler.step(train_loss)

    return model, train_loss, train_loss_history, val_loss_history

In [14]:
# ==========================================
# Sample weights: prevent months with more listed stocks from dominating
# ==========================================
def calculate_sample_weights(df_dates, mode='month_equal', half_life_months=36):
    dates = pd.to_datetime(df_dates).reset_index(drop=True)
    if mode == 'uniform':
        weights = np.ones(len(dates), dtype=np.float64)
    elif mode in {'month_equal', 'time_decay'}:
        month_counts = dates.groupby(dates).transform('size').to_numpy(dtype=float)
        weights = 1.0 / np.maximum(month_counts, 1.0)
        if mode == 'time_decay':
            max_date = dates.max()
            months_diff = (
                (max_date.year - dates.dt.year) * 12
                + (max_date.month - dates.dt.month)
            ).to_numpy(dtype=float)
            weights *= np.power(0.5, months_diff / half_life_months)
    else:
        raise ValueError("weights_type must be 'uniform', 'month_equal', or 'time_decay'")
    weights = weights / weights.mean()
    return weights.astype(np.float32)

In [ ]:
if __name__ == "__main__":
    seed_everything(seed=random_seed)

    # 1. 实例化时间窗口生成器。索引必须与全局 Tensor 一一对应。
    processed_df = processed_df.reset_index(drop=True)
    generator = DataWindowGenerator(df=processed_df, date_col='mthcaldt')
    if windows_type == 'expanding':
        window_gen = generator.expanding_window_generator(
            initial_train_months=initial_train_months,
            val_months=val_months,
            test_months=test_months,
        )
    else:
        window_gen = generator.rolling_window_generator(
            train_months=initial_train_months,
            val_months=val_months,
            test_months=test_months,
        )

    # "Pushing the full dataset into device memory once..."
    global_X_mac = torch.from_numpy(
        processed_df[macro_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)
    global_X_mic = torch.from_numpy(
        processed_df[micro_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)
    global_ind = torch.from_numpy(
        processed_df['ffi49'].to_numpy(dtype=np.int64, copy=True)
    ).to(device)
    global_Y = torch.from_numpy(
        processed_df['target_ret_final'].to_numpy(dtype=np.float32, copy=True)
    ).to(device)

    history_logs = []
    all_oos_predictions = []
    macro_dim = len(macro_cols)
    micro_dim = len(micro_cols)

    # 模型只初始化一次，rolling 窗口之间继续 warm-start
    if MODEL_TYPE == "fused":
        global_model = FusedMDN(
            macro_dim=macro_dim,
            micro_dim=micro_dim,
            num_industries=50,
            ind_emb_dim=6,
            hidden_dim=64,
            num_components=best_k,
            dist_type=GLOBAL_DIST_TYPE,
            dropout_rate=dropout_rate,
        ).to(device)

    elif MODEL_TYPE == "NoFiLM":
        global_model = StructuredMDN_NoFiLM(
            macro_dim=macro_dim,
            micro_dim=micro_dim,
            num_industries=50,
            ind_emb_dim=6,
            num_components=best_k,
            dist_type=GLOBAL_DIST_TYPE,
            dropout_rate=dropout_rate,
        ).to(device)

    elif MODEL_TYPE == 'FiLM':
        global_model = StructuredMDN_with_FiLM(
            macro_dim,
            micro_dim,
            num_industries=50,
            ind_emb_dim=6,
            num_components=best_k,
            dist_type=GLOBAL_DIST_TYPE,
            dropout_rate=dropout_rate,
        ).to(device)
    else:
        raise ValueError(
            "MODEL_TYPE 必须是 'fused' 或 'NoFiLM' 或 'FiLM'"
        )

    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(
            f"[Window {window_idx}] "
            f"Train: {info['train'][0]}~{info['train'][1]} | "
            f"Val: {info['val'][0]}~{info['val'][1]} | "
            f"Test: {info['test'][0]}~{info['test'][1]}"
        )
        start_time = time.time()

        # 提取各集合在全局大表中的行索引。
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # 直接切片预先计算好的标准化值。
        X_tr_mac = global_X_mac[idx_tr]
        X_va_mac = global_X_mac[idx_va]
        X_te_mac = global_X_mac[idx_te]

        X_tr_mic, X_tr_ind, Y_tr = global_X_mic[idx_tr], global_ind[idx_tr], global_Y[idx_tr]
        X_va_mic, X_va_ind, Y_va = global_X_mic[idx_va], global_ind[idx_va], global_Y[idx_va]
        X_te_mic, X_te_ind       = global_X_mic[idx_te], global_ind[idx_te]

        # Month-equal weighting prevents months with more firms from dominating.
        if epochs_type == 'fixed':
            combined_dates = pd.concat(
                [train_df['mthcaldt'], val_df['mthcaldt']], ignore_index=True
            )
            combined_weights = calculate_sample_weights(
                combined_dates, mode=weights_type, half_life_months=36
            )
            w_train_np = combined_weights[:len(train_df)]
            w_val_np = combined_weights[len(train_df):]
        else:
            w_train_np = calculate_sample_weights(
                train_df['mthcaldt'], mode=weights_type, half_life_months=36
            )
            w_val_np = calculate_sample_weights(
                val_df['mthcaldt'], mode=weights_type, half_life_months=36
            )

        w_train = torch.tensor(w_train_np, dtype=torch.float32, device=device)
        w_val = torch.tensor(w_val_np, dtype=torch.float32, device=device)

        current_lr = lr if window_idx == 0 else lr * 0.3

        is_first_window = (window_idx == 0)

        if epochs_type == 'fixed':
            global_model, best_v_loss, tr_hist, va_hist = train_structured_mdn_fixed_epochs(
                model=global_model,
                X_train_macro=X_tr_mac,
                X_train_micro=X_tr_mic,
                X_train_industry=X_tr_ind,
                y_train=Y_tr,
                w_train=w_train,
                X_val_macro=X_va_mac,
                X_val_micro=X_va_mic,
                X_val_industry=X_va_ind,
                y_val=Y_va,
                w_val=w_val,
                lr=current_lr,
                first_window_epochs=first_window_epochs,
                later_window_epochs=later_window_epochs,
                patience=patience,
                batch_size=batch_size,
                is_first_window=is_first_window,
            )
        else:
            global_model, best_v_loss, tr_hist, va_hist = train_structured_mdn(
                model=global_model,
                X_train_macro=X_tr_mac,
                X_train_micro=X_tr_mic,
                X_train_industry=X_tr_ind,
                y_train=Y_tr,
                w_train=w_train,
                X_val_macro=X_va_mac,
                X_val_micro=X_va_mic,
                X_val_industry=X_va_ind,
                y_val=Y_va,
                w_val=w_val,
                lr=current_lr,
                epochs=early_stopping_max_epochs,
                patience=patience,
                batch_size=batch_size,
            )

            # # 基于 Train + Val 联合集合做短暂 fine-tune。
            # epochs_run = len(tr_hist)
            # finetune_epochs = 2 if run_test == 'quick' else max(5, epochs_run // 10)
            # print(
            #     f"  Phase 1 done: epochs_run={epochs_run}, "
            #     f"finetune_epochs={finetune_epochs}"
            # )

            # X_tr_mac_ft = torch.cat([X_tr_mac, X_va_mac], dim=0)
            # X_tr_mic_ft = torch.cat([X_tr_mic, X_va_mic], dim=0)
            # X_tr_ind_ft = torch.cat([X_tr_ind, X_va_ind], dim=0)
            # Y_tr_ft = torch.cat([Y_tr, Y_va], dim=0)

            # train_val_df = pd.concat([train_df, val_df], ignore_index=True)
            # w_ft_np = calculate_sample_weights(
            #     train_val_df['mthcaldt'], mode=weights_type, half_life_months=36
            # )
            # w_ft = torch.tensor(w_ft_np, dtype=torch.float32, device=device)

            # global_model, _, _, _ = train_structured_mdn(
            #     model=global_model,
            #     X_train_macro=X_tr_mac_ft,
            #     X_train_micro=X_tr_mic_ft,
            #     X_train_industry=X_tr_ind_ft,
            #     y_train=Y_tr_ft,
            #     w_train=w_ft,
            #     lr=current_lr * 0.5,
            #     epochs=finetune_epochs,
            #     patience=finetune_epochs + 1,
            # )
            # print(
            #     f"  Phase 2 fine-tune done "
            #     f"({finetune_epochs} epochs on Train+Val)"
            # )

        duration = time.time() - start_time
        history_logs.append({
            'window': window_idx,
            'train_period': f"{info['train'][0]} to {info['train'][1]}",
            'test_period': f"{info['test'][0]} to {info['test'][1]}",
            'duration_sec': duration,
            'best_val_loss': best_v_loss,
            'epochs_run': len(tr_hist),
            'train_loss_trace': tr_hist,
            'val_loss_trace': va_hist,
        })

        # 样本外预测。
        global_model.eval()
        with torch.no_grad():
            pi_logits, mu, sigma, nu = global_model(
                X_te_mac, X_te_mic, X_te_ind
            )
            pi = torch.softmax(pi_logits, dim=-1)

        preds_df = test_df[['permno', 'mthcaldt', 'target_ret_final']].copy()
        preds_df["model_id"] = f"{MODEL_TYPE}_{GLOBAL_DIST_TYPE}_k{best_k}"
        preds_df['pi_vec'] = [
            json.dumps(vec.tolist()) for vec in pi.cpu().numpy()
        ]
        preds_df['pi_logits_vec'] = [
            json.dumps(vec.tolist()) for vec in pi_logits.cpu().numpy()
        ]
        preds_df['mu_vec'] = [
            json.dumps(vec.tolist()) for vec in mu.cpu().numpy()
        ]
        preds_df['sigma_vec'] = [
            json.dumps(vec.tolist()) for vec in sigma.cpu().numpy()
        ]

        if nu is not None:
            preds_df['nu_vec'] = [
                json.dumps(vec.tolist()) for vec in nu.cpu().numpy()
            ]
        else:
            preds_df['nu_vec'] = None

        all_oos_predictions.append(preds_df)

        if run_test == "quick" and window_idx == 1:
            break

    # 提取并保存行业 Embedding。
    embedding_weights = global_model.ind_embedding.weight.detach().cpu().numpy()
    df_emb = pd.DataFrame(
        embedding_weights,
        columns=[f'emb_dim_{i + 1}' for i in range(6)],
    )
    df_emb['ffi49'] = df_emb.index
    final_oos_df = pd.concat(all_oos_predictions, ignore_index=True)
    metrics_df = pd.DataFrame(history_logs)
    print(f"Successfully generated {len(final_oos_df)} predictions.")

Global seed set to 42 to ensure reproducibility.
[Window 0] Train: 1991-01~2005-12 | Val: 2006-01~2006-12 | Test: 2007-01~2007-12
[Window 1] Train: 1992-01~2006-12 | Val: 2007-01~2007-12 | Test: 2008-01~2008-12
Successfully generated 87375 predictions.


In [16]:
# 指定保存路径
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

if env == 'kaggle':
    path = '/kaggle/working'
elif env == 'colab':
    path = '/content/drive/MyDrive/Colab Notebooks/'
else:
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'

# 执行保存
final_oos_df.to_parquet(os.path.join(path, f'final_oos_df_{timestamp}.parquet'), index=False)
metrics_df.to_parquet(os.path.join(path, f'metrics_df_{timestamp}.parquet'), index=False)
df_emb.to_parquet(os.path.join(path, f'industry_embeddings_50x6_{timestamp}.parquet'), index=False)
print(f"保存成功！文件位置: {path}")

保存成功！文件位置: /Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output


In [17]:
if env == 'colab':
    # 运行完你的所有逻辑后，调用此命令会自动释放当前后端实例
    google.colab.runtime.unassign()